In [1]:
!nvidia-smi

Fri Apr 10 10:00:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile input.txt
Today's weather is sunny and warm. I love sunny days. The sun is shining brightly in the sky 

Writing input.txt


In [3]:
%%writefile wordcount.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <cuda.h>

#define MAX_WORDS 1000
#define MAX_LEN 20

// ---------------- GPU KERNEL ----------------
__global__ void countWords(char words[][MAX_LEN], int *counts, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n) {
        int count = 0;

        for (int j = 0; j < n; j++) {
            int match = 1;

            for (int k = 0; k < MAX_LEN; k++) {
                if (words[i][k] != words[j][k]) {
                    match = 0;
                    break;
                }
                if (words[i][k] == '\0') break;
            }

            if (match) count++;
        }

        counts[i] = count;
    }
}

// ---------------- LOWERCASE FUNCTION ----------------
void toLowerCase(char *word) {
    for (int i = 0; word[i]; i++) {
        word[i] = tolower(word[i]);
    }
}

// ---------------- MAIN ----------------
int main() {
    FILE *fp = fopen("input.txt", "r");
    if (!fp) {
        printf("File not found!\n");
        return 1;
    }

    char words[MAX_WORDS][MAX_LEN];
    int n = 0;

    // -------- READ FILE --------
    while (fscanf(fp, "%s", words[n]) != EOF) {
        toLowerCase(words[n]);
        n++;
    }
    fclose(fp);

    printf("Total words: %d\n", n);

    // -------- GPU MEMORY --------
    char (*d_words)[MAX_LEN];
    int *d_counts;
    int counts[MAX_WORDS];

    cudaMalloc(&d_words, sizeof(char) * MAX_WORDS * MAX_LEN);
    cudaMalloc(&d_counts, sizeof(int) * MAX_WORDS);

    cudaMemcpy(d_words, words, sizeof(char) * MAX_WORDS * MAX_LEN, cudaMemcpyHostToDevice);

    // -------- LAUNCH KERNEL --------
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    countWords<<<blocks, threads>>>(d_words, d_counts, n);

    cudaMemcpy(counts, d_counts, sizeof(int) * MAX_WORDS, cudaMemcpyDeviceToHost);

    // -------- PRINT DISTINCT WORDS --------
    printf("\nWord Count:\n");

    for (int i = 0; i < n; i++) {
        int printed = 0;

        // Check if already printed
        for (int j = 0; j < i; j++) {
            if (strcmp(words[i], words[j]) == 0) {
                printed = 1;
                break;
            }
        }

        if (!printed) {
            printf("%s : %d\n", words[i], counts[i]);
        }
    }

    // -------- CLEANUP --------
    cudaFree(d_words);
    cudaFree(d_counts);

    return 0;
}

Writing wordcount.cu


In [4]:
!nvcc wordcount.cu -o wordcount
!./wordcount

Total words: 18

Word Count:
today's : 1
weather : 1
is : 2
sunny : 2
and : 1
warm. : 1
i : 1
love : 1
days. : 1
the : 2
sun : 1
shining : 1
brightly : 1
in : 1
sky : 1
